# Free Web Search Optimized for LLMs

This notebook demonstrates completely free web search and content extraction optimized for LLM usage.

**Features:**
- **DuckDuckGo Search** (`ddgs`) - Free web search without API keys
- **Jina AI Reader** - Extract clean markdown from URLs (free, no API key)
- **DDGS Extract** - Built-in content extraction from URLs
- **Integration with OpenRouter** - Send search results to LLM

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from ddgs import DDGS
import requests

# Load environment variables
load_dotenv()

# Initialize OpenRouter client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

print("[OK] Setup complete!")

[OK] Setup complete!


## 1. DuckDuckGo Web Search

Search the web using DuckDuckGo. Completely free, no API key required.

In [2]:
def web_search(query, max_results=5):
    """
    Search the web using DuckDuckGo.
    Returns: list of dicts with 'title', 'href', 'body'
    """
    ddgs = DDGS()
    results = ddgs.text(query, max_results=max_results)
    return results

# Test search
search_results = web_search("latest AI developments 2024", max_results=3)

print(f"Found {len(search_results)} results\n")
for i, result in enumerate(search_results, 1):
    print(f"{i}. {result['title']}")
    print(f"   URL: {result['href']}")
    print(f"   Snippet: {result['body'][:150]}...\n")

Found 3 results

1. AI News | Latest News | Insights Powering AI-Driven Business Growth
   URL: https://www.artificialintelligence-news.com/
   Snippet: 2 days ago - AI News delivers the latest updates in artificial intelligence, machine learning, deep learning, enterprise AI, and emerging tech worldwi...

2. AI 2024 in review: The 10 most notable AI stories of the year
   URL: https://iot-analytics.com/ai-2024-10-most-notable-stories/
   Snippet: February 6, 2025 - On March 18, 2024, after years of hype about its A100- and H100-series data center GPUs, US-based chip designer and developer NVIDI...

3. Latest AI Breakthroughs and Their Impact in 2024
   URL: https://indiaai.gov.in/article/the-future-is-now-latest-ai-breakthroughs-and-their-impact-in-2024
   Snippet: ... The innovations in AI for 2024 are transforming industries, improving lives, and pushing the boundaries of what technology can achieve. From gener...



## 2. Extract Clean Content from URLs

Use Jina AI Reader (free, no API key) to extract clean markdown from any URL.

In [3]:
def extract_with_jina(url):
    """
    Extract clean markdown from URL using Jina AI Reader (free).
    Returns: clean markdown text
    """
    jina_url = f"https://r.jina.ai/{url}"
    response = requests.get(jina_url, timeout=30)
    if response.status_code == 200:
        return response.text
    else:
        return f"Error: {response.status_code}"

# Test with first search result
if search_results:
    test_url = search_results[0]['href']
    print(f"Extracting content from: {test_url}\n")
    content = extract_with_jina(test_url)
    print(content[:1000])  # Print first 1000 chars
    print("\n... [truncated]")

Extracting content from: https://www.artificialintelligence-news.com/

Title: AI News | Latest AI News, Analysis & Events

URL Source: https://www.artificialintelligence-news.com/

Markdown Content:
# AI News | Latest News | Insights Powering AI-Driven Business Growth

Manage Cookie Consent

We use technologies like cookies to store and/or access device information. We do this to improve browsing experience and to show personalized ads. Consenting to these technologies will allow us to process data such as browsing behavior or unique IDs on this site. Not consenting or withdrawing consent, may adversely affect certain features and functions.

Functional- [x] Functional  Always active 

The technical storage or access is strictly necessary for the legitimate purpose of enabling the use of a specific service explicitly requested by the subscriber or user, or for the sole purpose of carrying out the transmission of a communication over an electronic communications network.

Preferences- [

## 3. Alternative: DDGS Built-in Extract

DDGS also has built-in URL extraction (no external API needed).

In [4]:
def extract_with_ddgs(url, fmt="text_markdown"):
    """
    Extract content from URL using DDGS built-in extractor.
    fmt options: text_markdown, text_plain, text_rich, text, content
    """
    ddgs = DDGS()
    result = ddgs.extract(url, fmt=fmt)
    return result

# Test DDGS extract
if search_results:
    test_url = search_results[0]['href']
    print(f"Extracting with DDGS: {test_url}\n")
    ddgs_result = extract_with_ddgs(test_url)
    print(ddgs_result['content'][:1000] if 'content' in ddgs_result else str(ddgs_result)[:1000])
    print("\n... [truncated]")

Extracting with DDGS: https://www.artificialintelligence-news.com/

[Skip to content][1]

AI News is part of the TechForge Publications series
* [Explore All][2]
* [Developer][3]
* [IoT News][4]
* [MarketingTech][5]
* [CloudTech][6]
* [Telecoms][7]
* [TechHQ][8]
* [TechWire Asia][9]
* [Explore All][10]
* [Developer][11]
* [IoT News][12]
* [MarketingTech][13]
* [CloudTech][14]
* [Telecoms][15]
* [TechHQ][16]
* [TechWire Asia][17]

TechForge
* [News][18]
* [Categories][19]
  * [Physical AI][20]
  * [AI and Us][21]
    * [Environment & Sustainability][22]
    * [Human-AI Relationships][23]
    * [Open-Source & Democratised AI][24]
    * [Trust, Bias & Fairness][25]
    * [World of Work][26]
  * [AI in Action][27]
    * [Creative Industries][28]
    * [Cybersecurity AI][29]
    * [Education AI][30]
    * [Entertainment & Media][31]
    * [Finance AI][32]
    * [Government & Public Sector AI][33]
    * [Healthcare & Wellness AI][34]
    * [Legal Industry AI][35]
    * [Manufacturing & Engin

## 4. Complete Pipeline: Search -> Extract -> LLM

Full workflow that searches the web, extracts content, and sends to your LLM.

In [5]:
def search_and_extract(query, max_results=3, max_content_length=3000):
    """
    Complete pipeline: search web, extract content from top results.
    Returns: formatted context string for LLM
    """
    # Search
    results = web_search(query, max_results=max_results)
    
    context_parts = []
    context_parts.append(f"# Web Search Results for: '{query}'\n")
    
    for i, result in enumerate(results, 1):
        context_parts.append(f"\n## Source {i}: {result['title']}")
        context_parts.append(f"URL: {result['href']}\n")
        
        # Try to extract full content
        try:
            content = extract_with_jina(result['href'])
            # Truncate if too long
            if len(content) > max_content_length:
                content = content[:max_content_length] + "... [truncated]"
            context_parts.append(content)
        except Exception as e:
            # Fallback to snippet
            context_parts.append(result['body'])
            context_parts.append(f"\n[Note: Could not extract full content: {e}]")
    
    return "\n".join(context_parts), results

# Run the pipeline
query = "What are the latest developments in AI?"
context, sources = search_and_extract(query, max_results=2)

print("=" * 80)
print("EXTRACTED CONTEXT FOR LLM")
print("=" * 80)
print(context[:2000])
print("\n... [truncated for display]")

EXTRACTED CONTEXT FOR LLM
# Web Search Results for: 'What are the latest developments in AI?'


## Source 1: The State of AI: Global Survey 2025 | McKinsey
URL: https://www.mckinsey.com/capabilities/quantumblack/our-insights/the-state-of-ai

Title: Access Denied

URL Source: https://www.mckinsey.com/capabilities/quantumblack/our-insights/the-state-of-ai


Markdown Content:
You don't have permission to access "http://www.mckinsey.com/capabilities/quantumblack/our-insights/the-state-of-ai" on this server.
Reference #18.c9cf5868.1779051683.645ed473

https://errors.edgesuite.net/18.c9cf5868.1779051683.645ed473


## Source 2: What are the latest developments in AI as of February 2026? -
URL: https://askai.glarity.app/search/What-are-the-latest-developments-in-AI-as-of-February-2026

Title: What are the latest developments in AI as of February 2026?

URL Source: https://askai.glarity.app/search/What-are-the-latest-developments-in-AI-as-of-February-2026

Markdown Content:
Answered on:2026-02-

In [6]:
# Send to OpenRouter LLM with web context
def ask_llm_with_context(question, context, model="nvidia/nemotron-3-super-120b-a12b:free"):
    """
    Send question with web search context to LLM.
    """
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant. Use the provided web search context to answer the user's question. If the context doesn't contain enough information, say so."
        },
        {
            "role": "user",
            "content": f"Context from web search:\n\n{context}\n\nQuestion: {question}"
        }
    ]
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        extra_body={"reasoning": {"enabled": True}}
    )
    
    return response.choices[0].message

# Ask LLM with web context
print("[INFO] Asking LLM with web search context...\n")
llm_response = ask_llm_with_context(query, context)

print("[LLM Response]:")
print("=" * 80)
print(llm_response.content)

if hasattr(llm_response, 'reasoning_details') and llm_response.reasoning_details:
    print("\n[Reasoning Details]:")
    print("-" * 80)
    print(llm_response.reasoning_details)

[INFO] Asking LLM with web search context...

[LLM Response]:
Based on the most recent information available (as of February 2026), the AI landscape is seeing several noteworthy shifts:

1. **OpenAI’s Chip Concerns** – OpenAI has reportedly been dissatisfied with the performance of NVIDIA’s latest AI chips and is quietly exploring alternative solutions, even as discussions about potential investment collaborations between the two firms continue.

2. **Oracle’s Cloud‑Infrastructure Push** – Oracle is raising roughly $50 billion to expand its cloud infrastructure, a move aimed at meeting the growing compute demands of AI workloads—including those from OpenAI and xAI.

3. **AI in Robotics & Automation** – A “2026 AI Impacts Trend Observation” report highlights how AI is increasingly driving advances in robotics and manufacturing automation, promising transformative effects across multiple industrial sectors.

4. **Intensifying Chatbot Competition in China** – The market for conversational

## 5. News Search

Search specifically for news articles.

In [7]:
def search_news(query, max_results=5, timelimit="w"):
    """
    Search for news articles.
    timelimit: d (day), w (week), m (month)
    """
    ddgs = DDGS()
    results = ddgs.news(query, max_results=max_results, timelimit=timelimit)
    return results

# Search for recent AI news
news_results = search_news("artificial intelligence", max_results=3, timelimit="w")

print("Recent AI News:\n")
for i, article in enumerate(news_results, 1):
    print(f"{i}. {article['title']}")
    print(f"   Source: {article['source']}")
    print(f"   Date: {article.get('date', 'N/A')}")
    print(f"   URL: {article['url']}")
    print(f"   Summary: {article['body'][:200]}...\n")

Recent AI News:

1. My Top Artificial Intelligence Stock for Retirees (Hint: It's Not Nvidia)
   Source: AOL
   Date: 2026-05-17T16:09:00+00:00
   URL: https://www.aol.com/articles/top-artificial-intelligence-stock-retirees-152000630.html
   Summary: Retirees who want exposure to artificial intelligence (AI) often run into a basic conflict. The companies most closely associated with the theme tend to be volatile and expensive, and they pay little ...

2. Guest editorial: Taxing artificial intelligence would be a big mistake
   Source: Marin Independent Journal
   Date: 2026-05-17T19:57:00+00:00
   URL: https://www.marinij.com/2026/05/17/guest-editorial-taxing-artificial-intelligence-would-be-a-big-mistake/
   Summary: Artificial intelligence might be the most transformative technology ever devised. Exactly how its effects will work through the economy is impossible to say, but serious disruption of one kind or anot...

3. Vatican Sets Up Commission on Artificial Intelligence
   Source:

## 6. Image Search

Search for images (returns URLs).

In [8]:
def search_images(query, max_results=5):
    """Search for images."""
    ddgs = DDGS()
    results = ddgs.images(query, max_results=max_results)
    return results

# Search for images
image_results = search_images("cute puppies", max_results=3)

print("Image Results:\n")
for i, img in enumerate(image_results, 1):
    print(f"{i}. {img['title']}")
    print(f"   Image URL: {img['image']}")
    print(f"   Source: {img.get('source', 'N/A')}\n")

Image Results:

1. Cute Puppies Desktop Wallpaper Photos, Download The BEST Free Cute ...
   Image URL: https://images.pexels.com/photos/1108099/pexels-photo-1108099.jpeg?cs=srgb&dl=pexels-chevanon-1108099.jpg&fm=jpg
   Source: pexels.com

2. Cute Puppy Desktop Wallpapers on WallpaperDog
   Image URL: https://wallpaper.dog/large/17010569.jpg
   Source: wallpaper.dog

3. Cute Dogs and Puppies Wallpaper ·① WallpaperTag
   Image URL: https://wallpapertag.com/wallpaper/full/1/b/0/435688-free-cute-dogs-and-puppies-wallpaper-2560x1600-for-pc.jpg
   Source: wallpapertag.com



## Summary

You now have a complete free web search system:

| Feature | Function | Cost | API Key |
|---------|----------|------|---------|
| Web Search | `web_search()` | Free | No |
| URL Extraction | `extract_with_jina()` | Free | No |
| URL Extraction | `extract_with_ddgs()` | Free | No |
| News Search | `search_news()` | Free | No |
| Image Search | `search_images()` | Free | No |
| LLM Integration | `ask_llm_with_context()` | Free* | Yes (OpenRouter) |

*Using OpenRouter free tier model

**Optional upgrades:**
- **Tavily** (`tavily-python`) - More structured search, requires API key
- **Firecrawl** (`firecrawl-py`) - Better extraction, 1000 free credits/month

Add their API keys to `.env` if you want to use them.